# DS2002 Capstone — EV Charging Station Analytics

**Team Number:** `team-07`

**Team Members:**
- Austin Song (rtx2nb)
- Haero Lee (jva6yw)
- Nate Kim (sax6mw)

**Date:** 2026-03-30

---

## Cloud Setup

Authenticate to GCS, download raw data, and verify your team folder.

In [99]:
# Install the GCS library (run once)
# !pip install google-cloud-storage -q

In [100]:
import os
import json
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
client = storage.Client(project="ds2002-492012")
bucket = client.bucket("ds2002-capstone-sp26-v2")
print("GCS authentication successful.")
print(f"Bucket: {bucket.name}")

GCS authentication successful.
Bucket: ds2002-capstone-sp26-v2


In [101]:
# Download raw data from GCS
TEAM = "team-07"  # <-- change to your team number

files = [
    "raw-data/charging_sessions.csv",
    "raw-data/station_locations.csv",
    "raw-data/vehicle_types.csv",
    "raw-data/grid_operators.csv",
    "raw-data/energy_and_demand.db",
]

os.makedirs("data", exist_ok=True)
for f in files:
    blob = bucket.blob(f)
    local = os.path.join("data", os.path.basename(f))
    blob.download_to_filename(local)
    print(f"Downloaded {f}")

Downloaded raw-data/charging_sessions.csv
Downloaded raw-data/station_locations.csv
Downloaded raw-data/vehicle_types.csv
Downloaded raw-data/grid_operators.csv
Downloaded raw-data/energy_and_demand.db


## Data Loading

In [102]:
sessions = pd.read_csv("data/charging_sessions.csv")
stations = pd.read_csv("data/station_locations.csv")
vehicles = pd.read_csv("data/vehicle_types.csv")
operators = pd.read_csv("data/grid_operators.csv")

conn = sqlite3.connect("data/energy_and_demand.db")
demand = pd.read_sql("SELECT * FROM daily_demand_summary", conn)
grid = pd.read_sql("SELECT * FROM grid_capacity_levels", conn)
conn.close()

print(f"Sessions: {sessions.shape}")
print(f"Stations: {stations.shape}")
print(f"Vehicles: {vehicles.shape}")
print(f"Operators: {operators.shape}")
print(f"Demand:   {demand.shape}")
print(f"Grid:     {grid.shape}")

Sessions: (27451, 11)
Stations: (21, 8)
Vehicles: (42, 6)
Operators: (5, 7)
Demand:   (6570, 7)
Grid:     (1825, 6)


## Data Exploration

Inspect each dataset before cleaning.

In [103]:
dfs = {
    'sessions': sessions,
    'stations': stations,
    'vehicles': vehicles,
    'operators': operators,
    'demand': demand,
    'grid': grid
}

# shapes
print("-----df shapes-----")
for name, df in dfs.items():
    print(f"{name}: {df.shape}")

# dtypes
print("\n-----df dtypes-----")
for name, df in dfs.items():
    print(f"\n--{name}:--")
    for col, dtype in df.dtypes.items():
        print(f"  {col}: {dtype}")

# info
print("\n-----df info-----")
for name, df in dfs.items():
    print(f"\n--{name}:--")
    df.info()

# nulls
print("\n-----null counts-----")
for name, df in dfs.items():
    print(f"\n--{name}:--")
    print(df.isnull().sum().to_string())

-----df shapes-----
sessions: (27451, 11)
stations: (21, 8)
vehicles: (42, 6)
operators: (5, 7)
demand: (6570, 7)
grid: (1825, 6)

-----df dtypes-----

--sessions:--
  session_id: object
  station_id: object
  vehicle_id: object
  session_start: object
  session_end: object
  kwh_delivered: float64
  session_type: object
  cost_usd: object
  payment_method: object
  connector_used: object
  user_id: object

--stations:--
  station_id: object
  station_name: object
  city: object
  state: object
  zip_code: int64
  latitude: float64
  longitude: float64
  region: object

--vehicles:--
  vehicle_id: object
  vehicle_name: object
  vehicle_class: object
  connector_type: object
  battery_kwh: object
  manufacturer: object

--operators:--
  operator_id: object
  operator_name: object
  city: object
  state: object
  avg_daily_capacity_mw: int64
  peak_capacity_mw: object
  cost_per_kwh: float64

--demand:--
  date: object
  station_id: object
  total_sessions: int64
  total_kwh_delivered: 

## Data Cleaning Pipeline

Document every cleaning step. Show before and after.

In [119]:
# 0. Fix whitespace in all DataFrame columns first
for df_obj in [sessions, stations, vehicles, operators, demand, grid]:
    df_obj.columns = df_obj.columns.str.strip()

# 1. Clean Vehicles: standardize strings to fill missing battery_kwh
vehicles['vehicle_name_clean'] = vehicles['vehicle_name'].str.lower().str.strip()
vehicles['vehicle_class_clean'] = vehicles['vehicle_class'].str.lower().str.strip()

# Fix connector_type: replace 'css' with 'CCS'
vehicles['connector_type'] = vehicles['connector_type'].str.replace('css', 'CCS', case=False)

# create a reference map for battery capacity
cap_map = vehicles.dropna(subset=['battery_kwh']).set_index(['vehicle_name_clean', 'vehicle_class_clean'])['battery_kwh'].to_dict()

# fill missing values using the map
def fill_kwh(row):
    if pd.isna(row['battery_kwh']) or row['battery_kwh'] == '':
        return cap_map.get((row['vehicle_name_clean'], row['vehicle_class_clean']))
    return row['battery_kwh']

vehicles['battery_kwh'] = vehicles.apply(fill_kwh, axis=1)

# filter to only standard IDs (VH-###)
vehicles_cleaned = vehicles[vehicles['vehicle_id'].str.contains('^VH-', na=False)].copy()
vehicles_cleaned.drop(columns=['vehicle_name_clean', 'vehicle_class_clean'], inplace=True)

print("Cleaned Columns:", sessions.columns.tolist())
display(vehicles_cleaned.head(30))

Cleaned Columns: ['session_id', 'station_id', 'vehicle_id', 'session_start', 'session_end', 'kwh_delivered', 'session_type', 'cost_usd', 'payment_method', 'connector_used', 'user_id']


,vehicle_id,vehicle_name,vehicle_class,connector_type,battery_kwh,manufacturer
0,VH-001,Tesla Model 3,Sedan,CCS,75.0,Tesla
3,VH-002,Tesla Model Y,SUV,CCS,75.0,Tesla
6,VH-003,Tesla Model S,Sedan,CCS,100.0,Tesla
9,VH-004,Chevrolet Bolt EV,Hatchback,CCS,65.0,Chevrolet
12,VH-005,Chevrolet Bolt EUV,SUV,ccs,65.0,Chevrolet
15,VH-006,Ford Mustang Mach-E,SUV,CCS,91.0,Ford
18,VH-007,Ford F-150 Lightning,Truck,CCS,131.0,NaN
21,VH-008,Hyundai Ioniq 5,SUV,CCS,77.4,Hyundai
24,VH-009,Kia EV6,SUV,CCS,77.4,Kia
27,VH-010,Nissan Leaf,Hatchback,CHAdeMO,40.0,Nissan


## External API Integration

Pull weather or energy data and join with your charging data.

In [105]:
# Your API code here


---

## Question 1: Demand Surge Identification

> Which time periods experienced the greatest charging demand surges compared to the baseline?

In [106]:
# Your analysis for Q1


**Findings:** *(write your interpretation here)*

---

## Question 2: The Vehicle Consolidation Problem

> After standardizing all vehicle ID variants, what is the true daily charging volume by vehicle type?

In [107]:
# Your analysis for Q2


**Findings:** *(write your interpretation here)*

---

## Question 3: Weather and Grid Correlation

> How do temperature extremes correlate with daily charging demand and grid load?

In [108]:
# Your analysis for Q3


**Findings:** *(write your interpretation here)*

---

## Question 4: Station-Level Geographic Patterns

> Do all stations experience the same usage patterns?

In [109]:
# Your analysis for Q4


**Findings:** *(write your interpretation here)*

---

## Question 5: The Connector Type Investigation

> Is the CHAdeMO decline real, or a data artifact?

In [110]:
# Your analysis for Q5


**Findings:** *(write your interpretation here)*

---

## Cloud Upload

Upload your cleaned data back to your team folder in GCS.

In [111]:
# Upload cleaned files to GCS
# blob = bucket.blob(f"{TEAM}/cleaned_sessions.csv")
# blob.upload_from_filename("cleaned_sessions.csv")
# print("Uploaded.")


---

## Reflection

### 1. Data Quality Impact
*(your response)*

### 2. Cloud Pipeline Experience
*(your response)*

### 3. ETL Trade-offs
*(your response)*

### 4. Pipeline Trust
*(your response)*

### 5. Team Collaboration
*(your response)*